In [ ]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd
import pathlib
import os
import json


import networkx as nx
import numpy as np
from shapely.geometry import LineString, Point
import math
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

import numpy.typing as npt

from scipy.spatial import cKDTree
from shapely.strtree import STRtree

from shapely.geometry import LineString
import math

from utils.nx import snap_to_edges, build_final_path, filter_edges, convert_latlon, convert_mercator

In [ ]:
save_dir = pathlib.Path("/mnt/c/Users/nikita/qgisData/busroutes_new")
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
CITY_NAME = "Санкт-Петербург"
MERCATOR = 3857
WGS_84 = 4326
R_CONSOLIDATE_M = 12.0  # radius to fuse multi-node junctions (in meters)

TROLLEY_TYPE = "trolleybus"
TRAM_TYPE = "tram"
BUS_TYPE = "bus"

bus_parser = Parser.BusGraphParser(CITY_NAME)
tram_parser = Parser.TramGraphParser(CITY_NAME)
trolleybus_parser = Parser.TrolleyGraphParser(CITY_NAME)

all_routes = []

for route_info in trolleybus_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url, TROLLEY_TYPE))
for route_info in bus_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url, BUS_TYPE))
for route_info in tram_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url, TRAM_TYPE))

In [ ]:
def clean_route_name(route_name: str) -> str:
    # strip. remove repeated spaces
    cleaned_name = " ".join(route_name.strip().split())
    return cleaned_name


route_list = []
stops_list = []
for route_name, route_url, route_type in all_routes:
    print(route_name, route_url, route_type)
    route_coords_list = bus_parser.get_route(route_url)
    stops = bus_parser.get_stops(route_url)
    # swap lat and lon first
    rn = clean_route_name(route_name)
    for idx, route_coords in enumerate(route_coords_list):
        swapped_coords = [(lon, lat) for lat, lon in route_coords]
        route_list.append(dict(route_name=f"{rn} (idx={idx})", route_coords=swapped_coords, route_type=route_type))
    stops_list.append(dict(route_name=rn, stops=stops, route_type=route_type))

In [ ]:
route_list_geojson = []
for route in route_list:
    line = LineString(route["route_coords"])
    route_list_geojson.append(
        dict(
            type="Feature",
            properties=dict(route_name=route["route_name"], route_type=route["route_type"]),
            geometry=json.loads(shapely.to_geojson(line)),
        )
    )
geojson_data = dict(
    type="FeatureCollection", features=route_list_geojson, crs=dict(type="name", properties=dict(name="EPSG:4326"))
)
with open(save_dir / "all_routes.geojson", "w", encoding="utf-8") as f:
    f.write(json.dumps(geojson_data, ensure_ascii=False))

In [ ]:
stops_list_geojson = []
for route in stops_list:
    for stop in route["stops"]:
        point = Point(stop["long"], stop["lat"])
        stops_list_geojson.append(
            dict(
                type="Feature",
                properties=dict(route_name=route["route_name"], route_type=route["route_type"], stop_name=stop["name"]),
                geometry=json.loads(shapely.to_geojson(point)),
            )
        )
geojson_data = dict(
    type="FeatureCollection", features=stops_list_geojson, crs=dict(type="name", properties=dict(name="EPSG:4326"))
)
with open(save_dir / "all_stops.geojson", "w", encoding="utf-8") as f:
    f.write(json.dumps(geojson_data, ensure_ascii=False))

In [ ]:
with open("input/boundaries.geojson", "r") as f:
    geojson = f.read()
boundaries = shapely.from_geojson(geojson)
graph_raw = ox.graph_from_polygon(boundaries, network_type="drive", simplify=False)
graph_merc = ox.project_graph(graph_raw, to_crs=MERCATOR)
graph = ox.consolidate_intersections(graph_merc, tolerance=R_CONSOLIDATE_M, rebuild_graph=True)

nodes, edges = ox.graph_to_gdfs(graph)

In [ ]:
edges

In [ ]:
e_rtree = STRtree(edges["geometry"])
n_rtree = STRtree(nodes["geometry"])
nodes["stops"] = "[]"

In [ ]:
# match nodes and stops
# find closes edge to the stop
# pick the nearest node on that edge to the stop

for route in stops_list:
    for stop in route["stops"]:
        _stop_point_mercator = convert_latlon({"lon": stop["long"], "lat": stop["lat"]})
        stop_point_mercator = Point(_stop_point_mercator[0], _stop_point_mercator[1])
        nearest_edge_id = e_rtree.nearest(stop_point_mercator)
        # nearest_edge_geom = edges["geometry"].iloc[nearest_edge]
        # distance = stop_point_mercator.distance(nearest_edge_geom)

        # find nearest node on that edge
        nearest_edge = edges.iloc[nearest_edge_id]
        nearest_edge_nodes = [nearest_edge.name[0], nearest_edge.name[1]]
        nearest_node_idx = None
        min_node_distance = float("inf")
        for node_idx in nearest_edge_nodes:
            node_geom = nodes["geometry"].iloc[node_idx]
            node_distance = stop_point_mercator.distance(node_geom)
            if node_distance < min_node_distance:
                min_node_distance = node_distance
                nearest_node_idx = node_idx
        if nearest_node_idx is not None:
            # assign stop to this node
            existing_stops = nodes.at[nearest_node_idx, "stops"]
            stop_info = {
                "stop_name": stop["name"],
                "route_name": route["route_name"],
                "route_type": route["route_type"],
            }
            if existing_stops:
                existing_stops_list = json.loads(existing_stops)
                existing_stops_list.append(stop_info)
                nodes.at[nearest_node_idx, "stops"] = json.dumps(existing_stops_list, ensure_ascii=False)
            else:
                nodes.at[nearest_node_idx, "stops"] = json.dumps([stop_info], ensure_ascii=False)

In [ ]:
# match routes and edges
# find closest and matching in direction
edges["routes"] = "[]"

for route in route_list:
    route_series = gpd.GeoSeries([Point(x, y) for x, y in route["route_coords"]], crs=WGS_84).to_crs(epsg=MERCATOR)

    matched_edges = snap_to_edges(edges, e_rtree, route_series)
    filtered_edges = filter_edges(edges, matched_edges)
    final_path = build_final_path(graph, filtered_edges)

    route_info = {"route_name": route["route_name"]}

    for i in range(len(final_path) - 1):
        u = final_path[i]
        v = final_path[i + 1]
        # append route info if exists and not already in the list
        existing_routes = edges.at[(u, v, 0), "routes"]
        if existing_routes:
            existing_routes_list = json.loads(existing_routes)
            # if already in skip
            for existing_route in existing_routes_list:
                if existing_route["route_name"] == route_info["route_name"]:
                    continue
            existing_routes_list.append(route_info)
            edges.at[(u, v, 0), "routes"] = json.dumps(existing_routes_list, ensure_ascii=False)
        else:
            edges.at[(u, v, 0), "routes"] = json.dumps([route_info], ensure_ascii=False)

In [ ]:
with open(save_dir / "edges.geojson", "w") as f:
    f.write(edges.to_json())

with open(save_dir / "nodes.geojson", "w") as f:
    f.write(nodes.to_json())